<a href="https://colab.research.google.com/github/samithdhananjaya/phishing_site_urls/blob/IT25102012/almlpro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pandas numpy scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import re
from urllib.parse import urlparse

# 1. Dataset එක Load කරගැනීම
# Dataset path එක ඔයාගේ file location එකට අනුව වෙනස් කරන්න
df = pd.read_csv("phishing_site_urls.csv")

print("Dataset Shape:", df.shape)
print(df.head())


# 2. Target Variable එක Label Encode කිරීම (bad = 1, good = 0)
df["Label"] = df["Label"].map({"bad": 1, "good": 0})


# 3. URL එකෙන් Features Extract කරන Function එක සෑදීම
def extract_url_features(url):
    features = {}

    # Feature 1: URL එකේ මුළු දිග (Length of URL)
    features["url_length"] = len(str(url))

    # Feature 2-8: විශේෂ සංකේත ගණන (Counts of Special Characters)
    features["count_dot"] = url.count(".")
    features["count_hyphen"] = url.count("-")
    features["count_at"] = url.count("@")
    features["count_question"] = url.count("?")
    features["count_equal"] = url.count("=")
    features["count_slash"] = url.count("/")
    features["count_www"] = url.count("www")

    # Feature 9: HTTPS තිබේද යන්න (1 if present, 0 if not)
    features["has_https"] = 1 if "https" in str(url).lower() else 0

    # Feature 10: URL එකේ IP Address එකක් කෙලින්ම තියෙනවාද බලන්න
    ip_pattern = r"(([01]?\d\d?|2[0-4]\d|25[0-5])\.){3}([01]?\d\d?|2[0-4]\d|25[0-5])"
    features["has_ip"] = 1 if re.search(ip_pattern, str(url)) else 0

    # Feature 11: Digit (ඉලක්කම්) සංඛ්‍යාව
    features["count_digits"] = sum(c.isdigit() for c in str(url))

    # Feature 12: Alphabet (අකුරු) සංඛ්‍යාව
    features["count_letters"] = sum(c.isalpha() for c in str(url))

    return features


# 4. Feature Extraction ක්‍රියාවලිය Dataset එකට යෙදීම
print("\nExtracting Features... (This may take a few minutes)")

# Dynamic Extraction
extracted_features = df["URL"].apply(extract_url_features)
features_df = pd.DataFrame(list(extracted_features))

# Features සහ Original Label එක එකතු කිරීම
final_df = pd.concat([features_df, df["Label"]], axis=1)

print("\n--- Processed Dataset Sample ---")
print(final_df.head())

# 5. Clean & Extracted Data එක නව CSV එකකට Save කිරීම
final_df.to_csv("processed_phishing_data.csv", index=False)
print(
    "\nSaved processed data to 'processed_phishing_data.csv' successfully!"
)

Dataset Shape: (549346, 2)
                                                 URL Label
0  nobell.it/70ffb52d079109dca5664cce6f317373782/...   bad
1  www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...   bad
2  serviciosbys.com/paypal.cgi.bin.get-into.herf....   bad
3  mail.printakid.com/www.online.americanexpress....   bad
4  thewhiskeydregs.com/wp-content/themes/widescre...   bad

Extracting Features... (This may take a few minutes)

--- Processed Dataset Sample ---
   url_length  count_dot  count_hyphen  count_at  count_question  count_equal  \
0         225          6             4         0               1            4   
1          81          5             2         0               0            2   
2         177          7             1         0               0            0   
3          60          6             0         0               0            0   
4         116          1             1         0               1            0   

   count_slash  count_www  has_https  has_ip

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

# 1. Process කරගත් Dataset එක Load කිරීම
print("Loading processed dataset...")
df = pd.read_csv("processed_phishing_data.csv")

# Features (X) සහ Target (y) වෙන් කිරීම
X = df.drop("Label", axis=1)
y = df["Label"]

# 2. Data Standardize කිරීම (Logistic Regression සඳහා වැදගත්)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Train සහ Test Sets වලට බෙදීම (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f"Data Split: {X_train.shape[0]} train samples, {X_test.shape[0]} test samples.")

# 4. Model 1: Random Forest Classifier Initialization
print("\nTraining Random Forest Classifier...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Random Forest Predictions & Evaluation
rf_preds = rf_model.predict(X_test)
print("\n--- Random Forest Results ---")
print("Accuracy:", accuracy_score(y_test, rf_preds))
print("F1-Score:", f1_score(y_test, rf_preds))
print("Confusion Matrix:\n", confusion_matrix(y_test, rf_preds))
print("\nClassification Report:\n", classification_report(y_test, rf_preds))

# 5. Model 2: Logistic Regression Initialization
print("\nTraining Logistic Regression Classifier...")
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

# Logistic Regression Predictions & Evaluation
lr_preds = lr_model.predict(X_test)
print("\n--- Logistic Regression Results ---")
print("Accuracy:", accuracy_score(y_test, lr_preds))
print("F1-Score:", f1_score(y_test, lr_preds))
print("Confusion Matrix:\n", confusion_matrix(y_test, lr_preds))
print("\nClassification Report:\n", classification_report(y_test, lr_preds))

# 6. K-Fold Cross Validation (Optional Evaluation - 5 Folds)
print("\nPerforming K-Fold Cross Validation (5 Folds) on Random Forest...")
cv_scores = cross_val_score(rf_model, X_scaled, y, cv=5, scoring='accuracy')
print("Cross Validation Accuracy Scores:", cv_scores)
print("Average CV Accuracy:", cv_scores.mean())

Loading processed dataset...
Data Split: 439476 train samples, 109870 test samples.

Training Random Forest Classifier...

--- Random Forest Results ---
Accuracy: 0.866369345590243
F1-Score: 0.7419955716444663
Confusion Matrix:
 [[74076  4509]
 [10173 21112]]

Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.94      0.91     78585
           1       0.82      0.67      0.74     31285

    accuracy                           0.87    109870
   macro avg       0.85      0.81      0.83    109870
weighted avg       0.86      0.87      0.86    109870


Training Logistic Regression Classifier...

--- Logistic Regression Results ---
Accuracy: 0.7809229088923273
F1-Score: 0.43287309740351537
Confusion Matrix:
 [[76614  1971]
 [22099  9186]]

Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.97      0.86     78585
           1       0.82      0.29      0.43     31285

    accur